# Resume Parsing Framework Tutorial

Welcome to the Resume Parser tutorial. This notebook demonstrates how to use the framework to extract structured data (Name, Email, Skills) from PDF and Word documents.

##1- Install all the repository requirements

In [1]:
!git clone https://github.com/mhsefidgar/resume-parsing-framework.git
%cd resume-parsing-framework/
!pip install -e .

Cloning into 'resume-parsing-framework'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 51 (delta 14), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 27.03 KiB | 9.01 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/resume-parsing-framework
Obtaining file:///content/resume-parsing-framework
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.0/329.0 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 61.7 MB/s eta 0:00:00
  Building editable for resume_parser_project (pyproject.toml) ... done
  Created wheel for resume_parser_project: fil



## 2. Setup
First, ensure you have set up your `.env` file with `GEMINI_API_KEY`.

In [2]:
import os
import sys
import unittest
from dotenv import load_dotenv

# Add project root to path to ensure imports work
sys.path.append(os.path.abspath(".."))

from resume_parser.framework import ResumeParserFramework
from utils.api_validator import validate_gemini_api_key

# Load environment variables
load_dotenv("../.env")

if validate_gemini_api_key():
    print("✅ API Key Validated (using google-genai SDK)")
else:
    print("⚠️ API Key Invalid or Missing - Skills extraction might fail")

✅ API Key Validated (using google-genai SDK)


## 3. Initialize Framework
We create an instance of `ResumeParserFramework`. This sets up the parsers (PDF, Word) and extractors.

In [3]:
framework = ResumeParserFramework()

## 4. Generate Sample Data
If you don't have samples, let's generate them using our helper script.

In [4]:
from create_samples import create_sample_pdf, create_sample_docx

sample_pdf = "../sample_resumes/sample.pdf"
sample_docx = "../sample_resumes/sample.docx"

os.makedirs("../sample_resumes", exist_ok=True)
create_sample_pdf(sample_pdf)
create_sample_docx(sample_docx)

Created ../sample_resumes/sample.pdf
Created ../sample_resumes/sample.docx


## 5. Parse PDF Resume
We will now parse the PDF sample.

In [5]:
try:
    result_pdf = framework.parse_resume(sample_pdf)
    print("--- PDF Results ---")
    print(f"Name: {result_pdf.name}")
    print(f"Email: {result_pdf.email}")
    print(f"Skills: {result_pdf.skills}")
except Exception as e:
    print(f"Error: {e}")

--- PDF Results ---
Name: John Doe
Email: john.doe@example.com
Skills: ['Python', 'Machine Learning', 'Docker', 'Kubernetes']


### Convert to json object and save it

In [15]:
import os

json_path = "/content/json_dir"

if not os.path.exists(json_path):
    os.makedirs(json_path)
    print("Directory created!")
else:
    print("Directory already exist.")

Directory already exist.


In [19]:
from utils import parse_to_json

json_results = parse_to_json.convert_to_json(result_pdf)
print(json_results)

{
    "name": "John Doe",
    "email": "john.doe@example.com",
    "skills": [
        "Python",
        "Machine Learning",
        "Docker",
        "Kubernetes"
    ]
}


In [25]:
json_file_name = "PdfJson.json"
if os.path.exists(json_path):
  parse_to_json.save_resume_json(result_pdf, json_path + '/'+ json_file_name)
  print(f"JSON file saved to {json_path}")

JSON file saved to /content/json_dir


## 6. Parse Word Resume
Now let's parse the Word document.

In [6]:
try:
    result_word = framework.parse_resume(sample_docx)
    print("--- Word Results ---")
    print(f"Name: {result_word.name}")
    print(f"Email: {result_word.email}")
    print(f"Skills: {result_word.skills}")
except Exception as e:
    print(f"Error: {e}")

--- Word Results ---
Name: Jane Smith
Email: jane.smith@test.com
Skills: ['Java', 'Spring Boot', 'React', 'AWS']


### Convert to json object and save it

In [23]:
from utils import parse_to_json

json_results = parse_to_json.convert_to_json(result_word)
print(json_results)

{
    "name": "Jane Smith",
    "email": "jane.smith@test.com",
    "skills": [
        "Java",
        "Spring Boot",
        "React",
        "AWS"
    ]
}


In [24]:
json_file_name = "WordJson.json"
if os.path.exists(json_path):
  parse_to_json.save_resume_json(result_pdf, json_path + '/'+ json_file_name)
  print(f"JSON file saved to {json_path}")

JSON file saved to /content/json_dir


## 7. Integration Tests with Real Files
This section runs validation tests against the actual PDF and DOCX files generated earlier to ensure the parsers and extractors work together correctly.

In [ ]:
class TestNotebookRealFiles(unittest.TestCase):
    def setUp(self):
        self.pdf_path = "../sample_resumes/sample.pdf"
        self.docx_path = "../sample_resumes/sample.docx"
        self.framework = ResumeParserFramework()

    def test_pdf_parsing_integration(self):
        """Verify extraction from a real PDF file"""
        result = self.framework.parse_resume(self.pdf_path)
        self.assertEqual(result.name, "John Doe")
        self.assertEqual(result.email, "john.doe@example.com")
        print("✅ Real PDF Integration Test Passed")

    def test_docx_parsing_integration(self):
        """Verify extraction from a real DOCX file"""
        result = self.framework.parse_resume(self.docx_path)
        self.assertEqual(result.name, "Jane Smith")
        self.assertEqual(result.email, "jane.smith@test.com")
        print("✅ Real DOCX Integration Test Passed")

# Run tests
suite = unittest.TestLoader().loadTestsFromTestCase(TestNotebookRealFiles)
runner = unittest.TextTestRunner(verbosity=0)
result = runner.run(suite)

if result.wasSuccessful():
    print("\n🚀 All integration tests using real files passed!")
else:
    print("\n❌ Some integration tests failed.")

✅ Real DOCX Integration Test Passed


----------------------------------------------------------------------
Ran 2 tests in 2.664s

OK


✅ Real PDF Integration Test Passed

🚀 All integration tests using real files passed!


In [ ]:
!python /content/resume-parsing-framework/tests/test_speed.py

--- Speed Benchmark Test ---
🟡 Sample files not found. Attempting to generate...
Created /content/resume-parsing-framework/tests/sample_resumes/sample.pdf
Created /content/resume-parsing-framework/tests/sample_resumes/sample.docx
✅ Samples generated successfully.
Running 50 iterations for each parser...
PDF Parsing: 100% 50/50 [00:00<00:00, 632.94it/s]
DOCX Parsing: 100% 50/50 [00:01<00:00, 47.39it/s]

Results:
📄 PDF Avg Parse Time:  0.00183 sec/file
📝 DOCX Avg Parse Time: 0.02111 sec/file
--------------------------


In [ ]:
!python /content/resume-parsing-framework/tests/run_all_tests.py

--- Preparing Test Environment ---
Generating real PDF and DOCX samples...
Created sample_resumes/sample.pdf
Created sample_resumes/sample.docx
--- Running Unit Tests ---
test_email_extractor (test_extractors.TestExtractors.test_email_extractor) ... ok
test_name_extractor (test_extractors.TestExtractors.test_name_extractor) ... ok
test_orchestration_with_real_file (test_framework.TestFramework.test_orchestration_with_real_file) ... INFO:resume_parser.framework:Parsing file: sample_resumes/sample.pdf using PDFParser
INFO:resume_parser.framework:Extracting data from text...
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
ok
test_pdf_parser (test_parsers.TestParsers.test_pdf_parser) ... ok
test_word_parser (test_parsers.TestParsers.test_word_parser) ... ok

----------------------------------------------------------------------
Ran 5 te